# Auxiliar 10 - Aplicación de métodos de espacios de estados con `statsmodels.tsa.statespace` 

## Modelos y estimaciones

Muchos procesos, tales como los autoregresivos y de media móvil, pueden incorporarse en el formalismo más general de modelos en el espacio de estados. El paquete `statsmodels.tsa.statespace` de Python ofrece clases y métodos para resolver numéricamente un gran número de problemas que pueden ser llevados a la forma del espacio de estados, principalmente, mediante el uso de las ecuaciones de Kalman.

Para más detalles del paquete así como también para fijar las definiciones de las matrices usadas en los cálculos se puede consultar [Time Series Analysis by State Space Methods](https://www.statsmodels.org/stable/statespace.html). 

## Aplicación al modelo de Tendencia Local

Aquí presentamos un ejemplo conocido como de "Tendencia Local" que corresponde a un random walk con correccione por observaciones. El modelo se puede interpretar como un proceso de medición en el que sucesivas lecturas se incorporan buscando de refinar la estimación final. Es decir, las lecturas acarrean un error de observación $\epsilon$ y el modelo tipo "camino aleatorio" $x_t = x_{t-1} + \eta_t$ busca preservar la mejor estimación hasta el paso inmediatamente anterior a un nuevo registro.


Comenzamos generando una serie de observaciones. Escribimos la expresión general y luego reeemplazamos los valores de acuerdo a nuestro problema y las definiciones de las matrices dadas en [Time Series Analysis by State Space Methods](https://www.statsmodels.org/stable/statespace.html). 

Las observaciones son

$y_t = Z_t x_t + \epsilon_t$

Tomamos la matriz de diseño (' design') 

$Z_t=1$ con 

$\epsilon_t \sim \mathcal{N}(\mu=0,\sigma^2=0.5)$ 

resultando la matriz de covarianza de observaciones ('obs_cov') 

$H=[\sigma^2]=[0.5]$. 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import norm

# observaciones
N_obs = 100
rng = np.random.default_rng(12345)

sigma2v = 0.5
z = rng.normal(1, np.sqrt(sigma2v), N_obs)

y = pd.Series(z)

Ahora, el modelo es

$x_t = T x_{t-1} + R_t \eta_t$

Tomamos la matriz de transición ('transition') 

$T_t=1$ con  

$\eta_t \sim \mathcal{N}(\mu=0,\sigma^2=0.1)$ 

resultando la matriz de covarianza de estados ('state_cov') 

$Q=[\sigma^2]=[0.1]$ 

y la matriz de selección ('selection') $R=1$. 

Vamos a volcar estas expresiones en la definición de la clase. Dejamos las covarianzas de la observación y de los estados como parámetros para ajustar en el modelo (ver por ejemplo, [Representation in Python](https://www.chadfulton.com/fulton_statsmodels_2017/sections/3-python_representation.html) y [State space modeling: Local Linear Trends](https://www.statsmodels.org/stable/examples/notebooks/generated/statespace_local_linear_trend.html)).

In [ ]:
"""
Univariate Local Trend Model
"""

sigma2q=0.1
class LocalTrend(sm.tsa.statespace.MLEModel):
    def __init__(self, endog):
        # Model order
        k_states = k_posdef = 1 # dimesnión del espacio de estados

        # Initialize the statespace
        super(LocalTrend, self).__init__(
            endog,
            k_states=k_states,
            k_posdef=k_posdef,
            # initialization="approximate_diffuse",
            initialization="known",
            initial_state = np.array([0]),
            initial_state_cov = np.array([[0.5]]),
            loglikelihood_burn=k_states,
        )

        # Initialize the matrices
        self.ssm["design"] = np.array([1])
        self.ssm["transition"] = np.array([1])
        self.ssm["selection"] = np.array([1])

    @property
    def param_names(self):
        return ["sigma2.obs", "sigma2.model"]

    @property
    def start_params(self):
        # return [np.std(self.endog)] * 2
        return np.array([sigma2v, sigma2q])

    def transform_params(self, unconstrained):
        return unconstrained**2

    def untransform_params(self, constrained):
        return constrained**0.5

    def update(self, params, *args, **kwargs):
        # params = super(LocalTrend, self).update(params, *args, **kwargs)

        # Observation covariance
        self.ssm["obs_cov", 0, 0] = params[0]

        # State covariance
        self.ssm["state_cov", 0 , 0] = params[1]

Corremos el método de filtrado que aplica las ecuaciones de Kalman

In [ ]:
# Setup the model
mod = LocalTrend(y)

res = mod.filter([sigma2v, sigma2q])
# res = mod.fit()
print(res.summary())


Vamos a realizar una representación de las observaciones, de los valores *filtrados* y la salida `predicted_mean` con su intervalode confianza que, como veremos, solamente se adelanta un paso en el tiempo.

In [ ]:
# Perform prediction and forecasting
predict = res.get_prediction()
forecast = res.get_forecast(10)

# print the filtered estimate of the unobserved level
# print(res.filtered_state[0])         
# print(res.filtered_state_cov[0, 0]) 

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# Plot the results
y.plot(ax=ax, style="k.", label="Observations")
predict.predicted_mean.plot(ax=ax, label="One-step-ahead Prediction")
predict_ci = predict.conf_int(alpha=0.05)
predict_index = np.arange(len(predict_ci))
ax.fill_between(
    predict_index[2:], predict_ci.iloc[2:, 0], predict_ci.iloc[2:, 1], alpha=0.1
)

forecast.predicted_mean.plot(ax=ax, style="r", label="Forecast")
forecast_ci = forecast.conf_int()
forecast_index = np.arange(len(predict_ci), len(predict_ci) + len(forecast_ci))
ax.fill_between(
    forecast_index, forecast_ci.iloc[:, 0], forecast_ci.iloc[:, 1], alpha=0.1
)


filtered_2 = pd.Series(res.filtered_state[0])
filtered_2.plot(ax=ax, style="b", label="Filtered")


# Cleanup the image
legend = ax.legend(loc="lower left");

Ciertamente, el pronóstico no evoluciona en el tiempo ya que no ingresan nuevos datos. Al contrario, su error crece dado que no está controlado por nuevas observaciones. 

Vemos la evolución de la matriz de covarianza de estados y cómo su valor define el intervalo de confianza (al 95% que corresponde a 3 desviaciones estándares)

In [ ]:
plt.plot(res.filtered_state_cov[0, 0], label="Filtered State Covariance")
plt.plot(res.predicted_state_cov[0, 0], label="Predicted State Covariance")
plt.legend(loc="best")

Vemos que la diferencia entre 'Predicted State Covariance' y 'Filtered State Covariance' es justamente 0.1 que representa el error que agrega el modelo ('state_cov'). 

Además, aplicando de manera recursiva la fórmula para el error de lo que llamamos *análisis*

$$\sigma_{\mathrm{filtered}}^2 = \frac{1}{\frac{1}{\sigma_{\mathrm{obs}}^2} + \frac{1}{\sigma_{\mathrm{state}}^2}}$$

$$\sigma_{\mathrm{filtered}}^2 = \frac{1}{\frac{1}{0.5} + \frac{1}{\sigma_{\mathrm{filtered}}^2 +0.1}}$$

que para la iteración 100 alcanza un valor de aproxiamdamente 0.18. Notamos que debido a la ecuación para la evolución o predicción de la covarianza de estados $\mathbf{P} = \mathbf{T} \mathbf{P} \mathbf{T} + \mathbf{Q}$ la covarianza $\sigma_{\mathrm{filtered}}^2$ suma 0.1 (la covarianza del error del modelo).

Los intervalos de confianza para un nivel de significancia del 95%, se obtienen como $3\sqrt{\sigma_{\textrm{filtered}}^2}$

## Ejercicio

Utilizar el método `.fit()` para encontrar los valores de la covarianza de la observación y la covarianza del estado que resultan óptimos dados los valores de las observaciones. Descomentar la línea de `update` de parámetros, el método `.fit()` y usar una inicilización difusa. Analizar los valores obtenidos.

